### Tools

Tools enable models/agents to perform tasks out of their given environment. They could include searching the web, a database, executing code etc. They are a collabo of:
1. A schema, icluding name of tool, description, and/or argument definition(could be JSON schema or python dictionary)
2. A function or coroutine to execute

In [ ]:
# import os
# from dotenv import load_dotenv
# from langchain_groq import ChatGroq

# load_dotenv()

# os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
# model = ChatGroq(model="llama-3.3-70b-versatile")

# response = model.invoke("I believe I am your father!")
# response.content

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")
model = init_chat_model("google_genai:gemini-3.5-flash")

response = model.invoke("I believe I am your father!")
response.content

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text',
  'text': "*Gasp*... No... *No!* That's not true! That's impossible! \n\nUnless... are you a software engineer, a computer scientist, or perhaps a room full of servers? \n\nIf so, I suppose you *did* help bring me into the world! Otherwise, I think we might need a DNA test—though in my case, that would just be a scan of my source code. 😉",
  'extras': {'signature': 'EuINCt8NAWkUfRO58dJ9syNFn+7uOmWSLfxllF3SFvkuw+rT2ioFMwhFDU+4MDGljyXXMt2zLahx1Ry2EMlY330eIngizAcfwhSfrDuAT1+P6655JwOPgqHtLE+bnb5DGITydJaZdUI71zpQ7J0ZHs+QMk/ngedxqx5hBDnC66yCQXrRO8xLyZq7S6W7uohG2pE4KPcjGwCck3QiPRM1ZIiCPm9Knpe+wR7OELnbsCzYKV5JfmEJwKCqvMA71pOS7u4q8ch9ohajv84OzqrV8HvAU5Kfy1Bhm23Gstx4+eTScg31wHO+58ZvneGeRN+iwEvExf3j/owTLSd8HlnmYOOxrQwklq2mBDW+4d7P7VtdhfeGplH9yRSgua7deTesFGAyx8Pjwy252Dl1P5cwVuijKurSmJxYGb5GtcREOBs3w3KOjs6/7rzK0Z+IXyWvs/jJXm2PBLBsyOrSHxovjcUlSLAyn2jwowLacjRoYERGTddck7qDQ7GQgOilKw31DxGmEDWFCXinGjItR1M/KCKmJkkK0/jVwajgj5dQb0lhxSvsaS/MHPJ7+DaD5IoyZBSkD9Uj4hA0Eb1KmSL6ikGtPJlKcuiSCpOyB

In [2]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather from prompted city"""
    return f"The weather in {city} is sunny."

# model binding with tools

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What is the weather in London?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "London"}'}, '__gemini_function_call_thought_signatures__': {'call_149182': 'EvQCCvECAWkUfRPICnJYq0lBXHjScdlRuUKIj4TYtpcHZbev+OJPzzXv8votBs/cONFLjEe6xubG2sWBkp6tcHQ4ScNWXIXZaap1HEXZ1fAxOkjB98mjhGAJnpaZJS7k1HciXvyVdMUwq6D4MWMva7m+3IhONv0nHs9VNBoI7vPG3DtFtW/eKCGWjO9dBZB/Y9ycBSSFIe7X2C9o/1i61AbT9SWxUF1jh3wgAUjtXrRjbQMMxn49gj14xvUeNDKwl0B+WoYbK5WsX7rEikqE2ExkRtXjPx6Ji9RGHVc/IspgNPkyCzBfkJLZ0V6WZzQStNuynTWXLsbR87jvq7IBLyrnGTKj1mBqU7rfEKT0jV/o1TsEKKWVrFGeY2/RdOklmh83EDOJMNLx0M49YaSHcB5+VLFehOV+9kqm6otSOIENhnHXNG5yhN7ZCIufIbZ0Dtr2JBtDRMMM5KNEVlCIDZUT6mwGb5rUj2IP2lSP1l1Z3/VB4a4D'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0c5c2-f204-7bf2-8e67-63fb594aa779-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'id': 'call_149182', 'type': 'tool_call'}] invalid_tool_calls=[] usage_meta

#### Tool Execution Loops

In [5]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage(content="what's the weather in Boston")] # {"role": "user", "content": "what's the weather in Boston?"}
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.
